# 01 · Ingesta de vuelos de Brasil (ANAC VRA)

**Fuente:** ANAC, *Voo Regular Ativo* (VRA). Un archivo CSV por mes con cada vuelo regular que operó en Brasil:
empresa, número de vuelo, origen y destino (código ICAO), salida y llegada **previstas y reales** (hora de Brasília)
y situación del vuelo (realizado o cancelado).

**Qué hace este notebook:** descarga cada mes del periodo de `config/pipeline.yaml`, comprueba que el archivo sea un
VRA y lo guarda **sin modificar** en la capa bronze, junto con un manifiesto (tamaño, líneas, hash y URL de origen).

**Salida:** `bronze/anac_vra/year=AAAA/month=MM/VRA_AAAAM.csv` y `bronze/_manifiestos/anac_vra/`.

Si un mes ya está en bronze no se vuelve a descargar, así que el notebook se puede ejecutar varias veces sin duplicar datos.

In [1]:
import os
from pathlib import Path

import yaml

# Carpeta del repositorio (este notebook está en notebooks/)
RAIZ = Path.cwd().parent
with open(RAIZ / "config" / "pipeline.yaml", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)
tablas = cfg["lake"]["tablas"]

# En la EC2 la variable LAKE_BUCKET trae el nombre del bucket; en una laptop se trabaja sobre data/lake
BUCKET = os.environ.get("LAKE_BUCKET", "").strip()
RAIZ_LOCAL = RAIZ / cfg["lake"]["raiz_local"]
BASE = f"s3a://{BUCKET}" if BUCKET else RAIZ_LOCAL.resolve().as_uri()


def ruta(tabla: str) -> str:
    """Ruta completa de una tabla declarada en config/pipeline.yaml."""
    return f"{BASE}/{tablas[tabla]}"


def meses(desde: str, hasta: str) -> list[tuple[int, int]]:
    """Pares (año, mes) entre dos fechas AAAA-MM, ambas incluidas."""
    anio, mes = map(int, desde.split("-"))
    anio_fin, mes_fin = map(int, hasta.split("-"))
    salida = []
    while (anio, mes) <= (anio_fin, mes_fin):
        salida.append((anio, mes))
        anio, mes = (anio + 1, 1) if mes == 12 else (anio, mes + 1)
    return salida


DESDE, HASTA = cfg["periodo"]["inicio"], cfg["periodo"]["fin"]
print("Destino:", f"s3://{BUCKET}" if BUCKET else "data/lake (local)")
print("Periodo:", DESDE, "a", HASTA)

Destino: s3://latam-bigdata-lake-728fd64c
Periodo: 2023-01 a 2025-12


In [2]:
from urllib.parse import quote

import pandas as pd

fuente = cfg["fuentes"]["vra"]

## Funciones de apoyo

Descargar con reintentos, calcular la huella (hash SHA-256) de cada archivo, subirlo a bronze y dejar su manifiesto.

In [3]:
import hashlib
import json
import shutil
import tempfile
import time
from datetime import UTC, datetime

import boto3
import requests

s3 = boto3.client("s3") if BUCKET else None


def descargar(url: str, destino: Path, params: dict | None = None, reintentos: int = 5) -> bool:
    """Descarga por partes con reintentos de espera creciente. Devuelve False si el recurso no existe (404)."""
    cabeceras = {"User-Agent": cfg["fuentes"]["user_agent"]}
    for intento in range(1, reintentos + 1):
        try:
            with requests.get(url, params=params, headers=cabeceras, stream=True, timeout=300) as r:
                if r.status_code == 404:
                    return False
                r.raise_for_status()
                with open(destino, "wb") as f:
                    for bloque in r.iter_content(1 << 20):
                        f.write(bloque)
            return True
        except (requests.RequestException, OSError) as error:
            espera = min(5 * 2**intento, 300)
            print(f"  falló la descarga ({intento}/{reintentos}): {error}; reintento en {espera} s")
            time.sleep(espera)
    raise RuntimeError(f"No se pudo descargar {url}")


def sha256(ruta_archivo: Path) -> str:
    h = hashlib.sha256()
    with open(ruta_archivo, "rb") as f:
        for bloque in iter(lambda: f.read(1 << 20), b""):
            h.update(bloque)
    return h.hexdigest()


def contar_lineas(ruta_archivo: Path) -> int:
    with open(ruta_archivo, "rb") as f:
        return sum(bloque.count(b"\n") for bloque in iter(lambda: f.read(1 << 20), b""))


def existe(clave: str) -> bool:
    """True si la clave ya está en bronze."""
    if BUCKET:
        return s3.list_objects_v2(Bucket=BUCKET, Prefix=clave, MaxKeys=1).get("KeyCount", 0) > 0
    return (RAIZ_LOCAL / clave).exists()


def guardar(archivo: Path, clave: str) -> None:
    """Sube el archivo al bucket (o lo copia a data/lake en local)."""
    if BUCKET:
        s3.upload_file(str(archivo), BUCKET, clave)
    else:
        destino = RAIZ_LOCAL / clave
        destino.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(archivo, destino)


def escribir_manifiesto(fuente: str, clave: str, archivo: Path, url: str) -> dict:
    """Guarda de dónde salió el archivo, su tamaño, sus líneas, su hash y la fecha de descarga."""
    manifiesto = {
        "fuente": fuente, "archivo": clave, "url": url,
        "bytes": archivo.stat().st_size, "lineas": contar_lineas(archivo), "sha256": sha256(archivo),
        "descargado_utc": datetime.now(UTC).isoformat(timespec="seconds"),
    }
    temporal = archivo.with_name(archivo.name + ".json")
    temporal.write_text(json.dumps(manifiesto, ensure_ascii=False, indent=2), encoding="utf-8")
    guardar(temporal, f"{tablas['manifiestos']}/{fuente}/{archivo.name}.json")
    temporal.unlink()
    return manifiesto


def leer_manifiestos(fuente: str) -> list[dict]:
    """Todos los manifiestos de una fuente."""
    prefijo = f"{tablas['manifiestos']}/{fuente}/"
    if BUCKET:
        salida = []
        for pagina in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET, Prefix=prefijo):
            for objeto in pagina.get("Contents", []):
                salida.append(json.loads(s3.get_object(Bucket=BUCKET, Key=objeto["Key"])["Body"].read()))
        return salida
    return [json.loads(p.read_text(encoding="utf-8")) for p in sorted((RAIZ_LOCAL / prefijo).glob("*.json"))]

## 1. URL de cada mes

ANAC organiza los archivos por año y por mes en portugués; el nombre del archivo lleva el mes sin cero
(`VRA_20241.csv` es enero de 2024).

In [4]:
def url_mes(anio: int, mes: int) -> str:
    nombre_mes = quote(fuente["meses_pt"][mes - 1])
    return f"{fuente['url_base']}/{anio}/{mes:02d}%20-%20{nombre_mes}/VRA_{anio}{mes}.csv"


url_mes(2024, 1)

'https://sistemas.anac.gov.br/dadosabertos/Voos%20e%20opera%C3%A7%C3%B5es%20a%C3%A9reas/Voo%20Regular%20Ativo%20%28VRA%29/2024/01%20-%20Janeiro/VRA_20241.csv'

## 2. Validación del archivo

Un VRA válido empieza con la línea `Atualizado em: ...` y luego el encabezado de 12 columnas separadas por `;`.
Si ANAC cambia el formato, la ingesta se detiene aquí en lugar de guardar un archivo que después nadie podrá leer.

In [5]:
def validar(ruta: Path) -> None:
    with open(ruta, encoding="utf-8-sig") as f:
        primera = f.readline().strip()
        encabezado = f.readline().strip()
    if not primera.startswith("Atualizado"):
        raise ValueError(f"{ruta.name}: primera línea inesperada: {primera[:60]!r}")
    if "ICAO Empresa Aérea" not in encabezado or encabezado.count(";") != 11:
        raise ValueError(f"{ruta.name}: encabezado inesperado: {encabezado[:120]!r}")

## 3. Ingesta de un mes

In [6]:
def ingerir_mes(anio: int, mes: int, forzar: bool = False) -> dict | None:
    clave = f"{tablas['bronze_vra']}/year={anio}/month={mes:02d}/VRA_{anio}{mes}.csv"
    if existe(clave) and not forzar:
        print(f"{anio}-{mes:02d}: ya está en bronze")
        return None
    url = url_mes(anio, mes)
    with tempfile.TemporaryDirectory() as carpeta:
        archivo = Path(carpeta) / f"VRA_{anio}{mes}.csv"
        if not descargar(url, archivo):
            print(f"{anio}-{mes:02d}: ANAC no lo tiene publicado")
            return None
        validar(archivo)
        guardar(archivo, clave)
        manifiesto = escribir_manifiesto("anac_vra", clave, archivo, url)
    print(f"{anio}-{mes:02d}: {manifiesto['lineas']:,} líneas, {manifiesto['bytes'] / 1e6:.1f} MB")
    return manifiesto

## 4. Ejecutar el periodo completo

Tarda unos minutos: son 36 archivos de unos 12 MB.

In [7]:
for anio, mes in meses(DESDE, HASTA):
    ingerir_mes(anio, mes)

2023-01: ya está en bronze
2023-02: ya está en bronze
2023-03: ya está en bronze
2023-04: ya está en bronze
2023-05: ya está en bronze
2023-06: ya está en bronze
2023-07: ya está en bronze
2023-08: ya está en bronze
2023-09: ya está en bronze
2023-10: ya está en bronze
2023-11: ya está en bronze
2023-12: ya está en bronze


2024-01: ya está en bronze
2024-02: ya está en bronze
2024-03: ya está en bronze
2024-04: ya está en bronze
2024-05: ya está en bronze
2024-06: ya está en bronze
2024-07: ya está en bronze
2024-08: ya está en bronze
2024-09: ya está en bronze
2024-10: ya está en bronze
2024-11: ya está en bronze


2024-12: ya está en bronze
2025-01: ya está en bronze
2025-02: ya está en bronze
2025-03: ya está en bronze
2025-04: ya está en bronze
2025-05: ya está en bronze
2025-06: ya está en bronze
2025-07: ya está en bronze
2025-08: ya está en bronze
2025-09: ya está en bronze
2025-10: ya está en bronze
2025-11: ya está en bronze


2025-12: ya está en bronze


## 5. Verificación

Cada archivo de bronze tiene su manifiesto. La tabla muestra lo que quedó guardado.

In [8]:
resumen = pd.DataFrame(leer_manifiestos("anac_vra")).sort_values("archivo")
resumen["MB"] = (resumen["bytes"] / 1e6).round(1)
print(f"Archivos: {len(resumen)} · Líneas: {resumen['lineas'].sum():,} · Tamaño: {resumen['bytes'].sum() / 1e9:.2f} GB")
resumen[["archivo", "lineas", "MB", "descargado_utc"]]

Archivos: 36 · Líneas: 2,972,267 · Tamaño: 0.42 GB


,archivo,lineas,MB,descargado_utc
0,bronze/anac_vra/year=2023/month=01/VRA_20231.csv,87567,12.2,2026-11-22T03:35:27+00:00
4,bronze/anac_vra/year=2023/month=02/VRA_20232.csv,73597,10.3,2026-11-22T03:35:40+00:00
5,bronze/anac_vra/year=2023/month=03/VRA_20233.csv,82144,11.5,2026-11-22T03:35:56+00:00
6,bronze/anac_vra/year=2023/month=04/VRA_20234.csv,77350,10.8,2026-11-22T03:36:09+00:00
7,bronze/anac_vra/year=2023/month=05/VRA_20235.csv,81591,11.4,2026-11-22T03:36:22+00:00
8,bronze/anac_vra/year=2023/month=06/VRA_20236.csv,79600,11.1,2026-11-22T03:36:38+00:00
9,bronze/anac_vra/year=2023/month=07/VRA_20237.csv,87025,12.2,2026-11-22T03:37:05+00:00
10,bronze/anac_vra/year=2023/month=08/VRA_20238.csv,85167,11.9,2026-11-22T03:37:20+00:00
11,bronze/anac_vra/year=2023/month=09/VRA_20239.csv,80514,11.2,2026-11-22T03:37:34+00:00
1,bronze/anac_vra/year=2023/month=10/VRA_202310.csv,82514,11.5,2026-11-22T03:37:58+00:00
